In [ ]:
import numpy as np
import pandas as pd
import csv
import matplotlib.pyplot as plt
from decimal import Decimal,ROUND_FLOOR

%matplotlib inline
%matplotlib notebook

## 1.Load file.

In [ ]:
# load the csv file.
path = 'C:/Users/ZWX/PythonNotebooks/UWBM/Trial/'
InputData = pd.read_csv(path + 'input_csv.csv')

In [ ]:
date = InputData['date']
P_atm = InputData['P_atm']
Ref_grass = InputData['Ref.grass']
E_pot_OW = InputData['E_pot_OW']

## 2. Unsaturated zone ###

In [ ]:
iters = np.shape(P_atm)[0] # total timestep.

In [ ]:
UZ_measure = 0 # we do not consider 'measure' for the time being.

### 2.1 Assumptions

__1.__ The infiltration water from the open paved area flows directly to groundwater (percolation).

__2.__ The unsaturated zone area is equal to the unpaved area;

__3.__ Percolation to groundwater is limited by the saturated conductivity of the soil;


### 2.2 Build up using default settings in excel

 Using default settings in excel as a starting point.
 
a. ___I_up_uz, i.e. infiltration from unpaved area to unsaturated zone__, is an input. 

b. ___The groundwater level__ is also an input with which the interpolation of several factors in the module is done.

c. ___Other parameters__ are default. 

Because only the pavedroof has disconnected fration, r_up is only from pavedroof, so i_uz is only related to pavedroof. And we take this set of i_uz as input as the starting point to build up the unsaturated zone module.

__The soil type is 2, and the crop type is 1, root zone thickness is defined as 40 cm.__

#### 2.2.1 Modified SoilSelector and ETSelector####

In [ ]:
soilmatrix = pd.read_csv(path + 'soilparameter.csv')
etmatrix = pd.read_csv(path + 'ETparameter.csv')

In [ ]:
def ETSelector(a, b):
    #a soil type, b crop type
    sol = etmatrix.loc[(etmatrix.soil_type == int(a)) & (etmatrix.crop_type == int(b))]
    return sol

In [ ]:
# Selector is modified a bit from the "SoilSelector&ETSelector" to avoid multiple repeated reference of selector to improve efficiency.
def SoilSelector(a, b, c):
    #a soil type, b crop type, c GWL [m -MSL]
    if c>= 0.0 and c <= 2.5:
        c = float(Decimal(str(c)).quantize(Decimal('.1'), rounding=ROUND_FLOOR))
    elif c < 3.0:
        c = 2.5
    elif c < 5.0:
        c = int(c)
    elif c <= 10:
        c = 5.0
    else:
        c = 10.0
    rootzone_thickness = 100 * ETSelector(a, b)['th_rz_m'].values
    sol = soilmatrix.loc[(soilmatrix.soil_type == int(a)) & (soilmatrix.th_rz == int(rootzone_thickness)) & (soilmatrix.gwl == c)]
    return sol

In [ ]:
print(list(SoilSelector(2, 1, 1.5)))
SoilSelector(2, 1, 1.4)

#### 2.2.2 Input data preparation

 __(a). i_up_uz:__

In [ ]:
path = 'C:/Users/ZWX/PythonNotebooks/UWBM/Trial/'
c1s1 = pd.read_csv(path + 'sol/' + 'Results_Unpaved_c1s1.csv')
#print(list(c1s1))
i_uz = c1s1[' I_uz']

 __(b). E_ref:__

In [ ]:
e_ref = Ref_grass

__(c). gwl (from groundwater module):__

__Note that a very important stuff here:__  GWL is the input to the unsaturatedzone module calculated from the groudwater module. It is used to interpolate the theta_eq as well as the caprise_max. So when we prepare this data series from the excel, we have to make it in at least 8 digits. Otherwise, if in low digtis, the resulting theta_eq cannot be validated with excel results very much.

In [ ]:
gwl = [1.5000000000,1.5002020610,1.5004027312,1.5006029832,1.5008028132,1.5009144077,1.5008426741,1.5009553106,1.5010200653
,1.5012272762,1.5014333696,1.5016390315,1.5018000320,1.5019580838,1.5021158183,1.5023202649,1.5025240563,1.5027213291
,1.5029166936,1.5031116544,1.5033062055,1.5035003477,1.5036940819,1.5038874092,1.5040803304,1.5042728465,1.5044649583
,1.5046566669,1.5048479731,1.5050388779,1.5052297208,1.5054249016,1.5056196500,1.5058139897,1.5060079215,1.5062014463
,1.5063945651,1.5065872788,1.5067795883,1.5067397391,1.5068851625,1.5070730094,1.5072592447,1.5073966999,1.5075824937
,1.5077676672,1.5079524544,1.5080877041,1.5082720555,1.5084066431,1.5082979673,1.5081357780,1.5079741564,1.5078128447
,1.5076543392,1.5075310663,1.5074083632,1.5072858973,1.5071636695,1.5070416795]

(d). __theta_uz__: the initial value for theta_uz is coming from the initial_GWL = 1.5m

In [ ]:
theta_uz = np.zeros(iters)
theta_uz[0] = SoilSelector(2,1,1.5)['moist_cont_eq_ rz[mm]'].values
print(theta_uz[0])

#### 2.2.2 Using Arrays to build up the structure first.

In [ ]:
delta_t = 1 / 24

# related to soil type, should be lookedup in database, but is predefined here for module buildup.
et= ETSelector(2,1)
# reduction point
theta_h3l = et['theta_h3l_mm'].values
theta_h3h = et['theta_h3h_mm'].values
# complete saturation
theta_h1 = et['theta_h1_mm'].values
# field capacity
theta_h2 = et['theta_h2_mm'].values
# wilting point
theta_h4 = et['theta_h4_mm'].values

# 
i_up_uz = np.zeros(iters)
theta_h3 = np.zeros(iters)
r_meas = np.zeros(iters)
t_alpha = np.zeros(iters)
t_atm = np.zeros(iters)
gwl_up = np.zeros(iters)
gwl_low = np.zeros(iters)
theta_eq = np.zeros(iters)
capris_max = np.zeros(iters)
p_gw = np.zeros(iters)
k_sat = 67.9

t = 1 

while t <= iters - 1:
    
    # i_up_uz
    i_up_uz[t] = i_uz[t]
    
    # theta_h3
    # Equilibrium moisture content in the root zone at which reduction of transpiration starts 
    if e_ref[t] / (2 * delta_t) < 1:
        
        theta_h3[t] = theta_h3l
    
    elif e_ref[t] / (2 * delta_t) > 5:
        
        theta_h3[t] = theta_h3h
    
    else:
        
        theta_h3[t] = theta_h3l + (e_ref[t] / (2 * delta_t) - 1) / 4 * (theta_h3h - theta_h3l) 
    
    # t_alpha
    # Transpiration factor [-] for the current time step. Involved in theta_uz.
    if theta_uz[t-1] + i_up_uz[t] + r_meas[t] > theta_h1:
        
        t_alpha[t] = 0
    
    elif theta_uz[t-1] + i_up_uz[t] + r_meas[t] > theta_h2:
    
        t_alpha[t] = 1 - ((theta_uz[t-1] + i_up_uz[t] + r_meas[t]) - theta_h2) / (theta_h1 - theta_h2)
        
    elif theta_uz[t-1] + i_up_uz[t] + r_meas[t] > theta_h3[t]:
    
        t_alpha[t] = 1
        
    elif theta_uz[t-1] + i_up_uz[t] + r_meas[t] > theta_h4:
    
        t_alpha[t] = 1 - ((theta_uz[t-1] + i_up_uz[t] + r_meas[t]) - theta_h4) / (theta_h3[t] - theta_h4)
    
    else:
        t_alpha[t] = 0
        
    # t_atm 
    # transipration from unsturated zone to atmosphere during the current time step.
    t_atm[t] = e_ref[t] * t_alpha[t]
    
    # gwl_up
    # Note that I modify the function of the soil selector a bit in order to avoild multiple repeatitive calling selector
    c = gwl[t-1]
    if c>= 0.0 and c <= 2.5:
        c = float(Decimal(str(c)).quantize(Decimal('.1'), rounding=ROUND_FLOOR))
    elif c < 3.0:
        c = 2.5
    elif c < 5.0:
        c = int(c)
    elif c <= 10:
        c = 5.0
    else:
        c = 10.0
    gwl_up[t] = c
    
    # gwl_low
    # Next value to the gwl_up in table
    if gwl_up[t] < 2.5:
        gwl_low[t] = gwl_up[t] + 0.1
    elif gwl_up[t] < 3:
        gwl_low[t] = 3
    elif gwl_up[t] < 4:
        gwl_low[t] = 4
    elif gwl_up[t] < 5:
        gwl_low[t] = 5
    else:
        gwl_low[t] = 10
    
    # theta_eq
    # Equilibrium soil moisture content in the root zone for the current time step [mm].
    if gwl[t-1] < 10:
        theta_eq[t] = SoilSelector(2, 1, gwl_low[t])['moist_cont_eq_ rz[mm]'].values +(gwl_low[t] - gwl[t-1]) / (gwl_low[t] - gwl_up[t]) * (SoilSelector(2, 1, gwl_up[t])['moist_cont_eq_ rz[mm]'].values-SoilSelector(2, 1, gwl_low[t])['moist_cont_eq_ rz[mm]'].values)
    else:
        theta_eq[t] = SoilSelector(2, 1, 10)['moist_cont_eq_ rz[mm]'].values
    
    # capris_max
    # Maximum capillary rise for the current time step [mm/d].
    if gwl[t-1] < 10:
        capris_max[t] = SoilSelector(2, 1, gwl_low[t])['capris_max[mm/d]'].values +(gwl_low[t] - gwl[t-1]) / (gwl_low[t] - gwl_up[t]) * (SoilSelector(2, 1, gwl_up[t])['capris_max[mm/d]'].values-SoilSelector(2, 1, gwl_low[t])['capris_max[mm/d]'].values)
    else:
        capris_max[t] = SoilSelector(2, 1, 10)['capris_max[mm/d]'].values
    
    
    # p_gw
    # Percolation from the unsaturated zone to the groundwater for the current time step [mm].
    if theta_uz[t-1] + i_up_uz[t] + r_meas[t] - t_atm[t] > theta_eq[t]:
        
        p_gw[t] = min(theta_uz[t-1] + i_up_uz[t] + r_meas[t] - t_atm[t] - theta_eq[t], delta_t * k_sat)
        
    else:
        
        p_gw[t] = -1 * min(theta_eq[t] - (theta_uz[t-1] + i_up_uz[t] + r_meas[t] - t_atm[t]), delta_t * capris_max[t])
    
    
    # theta_uz
    theta_uz[t] = theta_uz[t-1] + i_up_uz[t] + r_meas[t] - t_atm[t] - p_gw[t]
    t += 1

filename = 'Results_UnsaturatedZone_arraybuildup.csv'
np.savetxt('sol/' + filename, np.c_[i_up_uz, theta_h3, t_alpha, t_atm, gwl_up, gwl_low, theta_eq, capris_max, p_gw, theta_uz], fmt = "%.8f", delimiter=',', header = 'i_up_uz, theta_h3, t_alpha, t_atm, gwl_up, gwl_low, theta_eq, capris_max, p_gw, theta_uz') 

# Insert the Date column for locating purposes.
df = pd.read_csv('sol/' + filename)
df.insert(0, 'Date', date)
df.to_csv('sol/' + filename)

print('The results have been validated with excel.')

#### 2.2.3 Using Class to build up the module. (C1S1)
The soil type is 2, and the crop type is 1, root zone thickness is 40 cm.

In [ ]:
class UnsaturatedZone:
    def __init__(self, theta_uz_t0, soiltype = 2, croptype = 1):
        
        # state
        self.init_theta_uz = theta_uz_t0
        
        # parameter
        
        self.soiltype = soiltype
        self.croptype = croptype      
    
        et= ETSelector(self.soiltype, self.croptype)
        self.theta_h3l = et['theta_h3l_mm'].values
        self.theta_h3h = et['theta_h3h_mm'].values
        self.theta_h1 = et['theta_h1_mm'].values
        self.theta_h2 = et['theta_h2_mm'].values
        self.theta_h4 = et['theta_h4_mm'].values
        
        self.k_sat = 10 * SoilSelector(self.soiltype, self.croptype, 1.5)['k_sat'].values # input gwl 1.5 does not affect the K_sat, which is only dependent on soiltype.
    
    def __repr__(self):
        return 'Current P is ' + str(p_atm) + 'Current E is ' + str(e_pot_ow) + '.These are current precipitation and evaporation.'
    
    def sol(self, i_uz, r_meas, e_ref, prev_gwl, delta_t = 1 / 24): 
        
        # i_up_uz
        i_up_uz = i_uz
        
        # theta_h3
        if e_ref / (2 * delta_t) < 1:
        
            theta_h3 = self.theta_h3l
    
        elif e_ref / (2 * delta_t) > 5:
        
            theta_h3 = self.theta_h3h
    
        else:
        
            theta_h3 = self.theta_h3l + (e_ref / (2 * delta_t) - 1) / 4 * (self.theta_h3h - self.theta_h3l) 
            
        # t_alpha
        if self.init_theta_uz + i_up_uz + r_meas > self.theta_h1:
        
            t_alpha = 0
    
        elif self.init_theta_uz + i_up_uz + r_meas > self.theta_h2:
    
            t_alpha = 1 - ((self.init_theta_uz + i_up_uz + r_meas) - self.theta_h2) / (self.theta_h1 - self.theta_h2)
        
        elif self.init_theta_uz + i_up_uz + r_meas > theta_h3:
    
            t_alpha = 1
        
        elif self.init_theta_uz + i_up_uz + r_meas > self.theta_h4:
    
            t_alpha = 1 - ((self.init_theta_uz + i_up_uz + r_meas) - self.theta_h4) / (theta_h3 - self.theta_h4)
    
        else:
            t_alpha = 0
            
        # t_atm 
        t_atm = e_ref * t_alpha
    
        # gwl_up
        c = prev_gwl
        
        if c>= 0.0 and c <= 2.5:
            c = float(Decimal(str(c)).quantize(Decimal('.1'), rounding=ROUND_FLOOR))
        elif c < 3.0:
            c = 2.5
        elif c < 5.0:
            c = int(c)
        elif c <= 10:
            c = 5.0
        else:
            c = 10.0
        gwl_up = c
            
        # gwl_low
        if gwl_up< 2.5:
            gwl_low = gwl_up + 0.1
        elif gwl_up < 3:
            gwl_low = 3
        elif gwl_up < 4:
            gwl_low = 4
        elif gwl_up < 5:
            gwl_low = 5
        else:
            gwl_low = 10
    
        # theta_eq
        if prev_gwl < 10:
            theta_eq = SoilSelector(self.soiltype, self.croptype, gwl_low)['moist_cont_eq_ rz[mm]'].values +(gwl_low - prev_gwl) / (gwl_low - gwl_up) * (SoilSelector(self.soiltype, self.croptype, gwl_up)['moist_cont_eq_ rz[mm]'].values-SoilSelector(self.soiltype, self.croptype, gwl_low)['moist_cont_eq_ rz[mm]'].values)
        else:
            theta_eq = SoilSelector(self.soiltype, self.croptype, 10)['moist_cont_eq_ rz[mm]'].values
        
        # capris_max
        if prev_gwl < 10:
            capris_max = SoilSelector(self.soiltype, self.croptype, gwl_low)['capris_max[mm/d]'].values +(gwl_low - prev_gwl) / (gwl_low - gwl_up) * (SoilSelector(self.soiltype, self.croptype, gwl_up)['capris_max[mm/d]'].values-SoilSelector(self.soiltype, self.croptype, gwl_low)['capris_max[mm/d]'].values)
        else:
            capris_max = SoilSelector(self.soiltype, self.croptype, 10)['capris_max[mm/d]'].values
            
        # p_gw
        if self.init_theta_uz + i_up_uz + r_meas - t_atm > theta_eq:
        
            p_gw = min(self.init_theta_uz + i_up_uz + r_meas - t_atm - theta_eq, delta_t * self.k_sat)
        
        else:
        
            p_gw = -1 * min(theta_eq - (self.init_theta_uz + i_up_uz + r_meas - t_atm), delta_t * capris_max)
    
    
        # theta_uz
        theta_uz = self.init_theta_uz + i_up_uz + r_meas - t_atm - p_gw
        
        # update state
        self.init_theta_uz = theta_uz

        
        return i_up_uz, theta_h3, t_alpha, t_atm, gwl_up, gwl_low, theta_eq, capris_max, p_gw, theta_uz

In [ ]:
t = 1

i_up_uz = [0]
theta_h3 = [0]
t_alpha= [0] 
t_atm = [0] 
gwl_up = [0] 
gwl_low = [0]
theta_eq = [0] 
capris_max = [0]
p_gw = [0]


# Give initial theta_uz.
theta_uz_t0 = SoilSelector(2, 1, 1.5)['moist_cont_eq_ rz[mm]'].values # 1.5m is initial gwl.
theta_uz = [theta_uz_t0]

# Specify the parameter or use the default setting.
m = UnsaturatedZone(theta_uz_t0, soiltype = 2, croptype = 1)

while t <= iters-1:
    # only loop sol(), not repeat creating new object.
    sol = m.sol(i_uz[t], r_meas[t], e_ref[t], prev_gwl = gwl[t-1], delta_t = 1/24)
    
    
    i_up_uz.append(sol[0])
    theta_h3.append(sol[1])
    t_alpha.append(sol[2]) 
    t_atm.append(sol[3]) 
    gwl_up.append(sol[4])
    gwl_low.append(sol[5])
    theta_eq.append(sol[6])
    capris_max.append(sol[7])
    p_gw.append(sol[8])
    theta_uz.append(sol[9])

    # print('time step', t)
    t += 1
    
filename = 'Results_UnsaturatedZone_c1s1.csv'
np.savetxt('sol/' + filename, np.c_[i_up_uz, theta_h3, t_alpha, t_atm, gwl_up, gwl_low, theta_eq, capris_max, p_gw, theta_uz], fmt = "%.8f", delimiter=',', header = 'i_up_uz, theta_h3, t_alpha, t_atm, gwl_up, gwl_low, theta_eq, capris_max, p_gw, theta_uz') 

# Insert the Date column for locating purposes.
df = pd.read_csv('sol/' + filename)
df.insert(0, 'Date', date)
df.to_csv('sol/' + filename)

print('The results have been validated.')

### 2.3 Validation using different coeffcient sets

#### 2.3.1 parameter set 1(C2S1):
__soiltype = 5, croptype = 1, init_gwl = 1.3m -MSL,__ rootzone thickness then is defined as 0.4m

In [ ]:
SoilSelector(5, 1, 1.3)

ETSelector(5,1)

theta_uz_t0 = SoilSelector(5, 1, 1.3)['moist_cont_eq_ rz[mm]'].values
print(theta_uz_t0)

##### a. Input data preparation: #####
__(a).i_up_uz__: it is the same as c1s1 because it is independent of parameters.

In [ ]:
i_uz = c1s1[' I_uz']

 __(b). E_ref:__

In [ ]:
e_ref = Ref_grass

__(c).gwl (from groundwater module)__

In [ ]:
gwl=[1.3000000000,1.3006641574,1.3013224097,1.3019788977,1.3026336025,1.3031904821,1.3035458386,1.3041019527,1.3046042630
,1.3052610777,1.3059151554,1.3065674692,1.3071697762,1.3077676968,1.3083640455,1.3090100618,1.3096540245,1.3102896271,1.3109219090
,1.3115525076,1.3121814191,1.3128086498,1.3134342057,1.3140580931,1.3146803179,1.3153008862,1.3159198041,1.3165370775,1.3171527124
,1.3177667147,1.3183794570,1.3189957064,1.3196102924,1.3202232524,1.3208345921,1.3214443172,1.3220524337,1.3226589472,1.3232638635
,1.3236170708,1.3241695504,1.3247664189,1.3253603514,1.3259005801,1.3264918679,1.3270812932,1.3276691813,1.3282026429,1.3287879260
,1.3293185171,1.3295866116,1.3297899998,1.3299929001,1.3301953327,1.3303999784,1.3306416496,1.3308866139,1.3311428698,1.3313984534
,1.3316534382]

In [ ]:
t = 1


i_up_uz = [0]
theta_h3 = [0]
t_alpha= [0] 
t_atm = [0] 
gwl_up = [0] 
gwl_low = [0]
theta_eq = [0] 
capris_max = [0]
p_gw = [0]


# Give initial theta_uz.
theta_uz_t0 = SoilSelector(5, 1, 1.3)['moist_cont_eq_ rz[mm]'].values # 1.5m is initial gwl.
theta_uz = [theta_uz_t0]

# Specify the parameter or use the default setting.
m = UnsaturatedZone(theta_uz_t0, soiltype = 5, croptype = 1)

while t <= iters-1:
    # only loop sol(), not repeat creating new object.
    sol = m.sol(i_uz[t], r_meas[t], e_ref[t], prev_gwl = gwl[t-1], delta_t = 1/24)
    
    
    i_up_uz.append(sol[0])
    theta_h3.append(sol[1])
    t_alpha.append(sol[2]) 
    t_atm.append(sol[3]) 
    gwl_up.append(sol[4])
    gwl_low.append(sol[5])
    theta_eq.append(sol[6])
    capris_max.append(sol[7])
    p_gw.append(sol[8])
    theta_uz.append(sol[9])

    # print('time step', t)
    t += 1
    
filename = 'Results_UnsaturatedZone_c2s1.csv'
np.savetxt('sol/' + filename, np.c_[i_up_uz, theta_h3, t_alpha, t_atm, gwl_up, gwl_low, theta_eq, capris_max, p_gw, theta_uz], fmt = "%.8f", delimiter=',', header = 'i_up_uz, theta_h3, t_alpha, t_atm, gwl_up, gwl_low, theta_eq, capris_max, p_gw, theta_uz') 

# Insert the Date column for locating purposes.
df = pd.read_csv('sol/' + filename)
df.insert(0, 'Date', date)
df.to_csv('sol/' + filename)

print('The results have been validated.')

#### 2.3.2 parameter set 2(C2S2):

__soiltype = 7, croptype = 1, init_gwl = 1.7m -MSL,__ rootzone thickness then is automatically defined as 0.3m due to the fixed combination in database.

__(a).i_up_uz__: and  __(b). E_ref:__ are the same as before pairs.

__(c).gwl (from groundwater module)__:

In [ ]:
gwl = [1.7000000000,1.6998124883,1.6996257957,1.6994394449,1.6992534364,1.6989862355,1.6985493185,1.6982838818,1.6979750608
,1.6977948101,1.6976148938,1.6974353111,1.6972312807,1.6970127421,1.6967946416,1.6966165806,1.6964388500,1.6962614491,1.6960843775,1.6959076345
,1.6957282829,1.6955490966,1.6953702403,1.6951917131,1.6950135143,1.6948356435,1.6946580999,1.6944808831,1.6943039925,1.6941274274
,1.6939515026,1.6937787194,1.6936062571,1.6934341152,1.6932622931,1.6930907901,1.6929196058,1.6927487395,1.6925781907,1.6922061068
,1.6919941956,1.6918229293,1.6916509301,1.6914341132,1.6912629474,1.6910919823,1.6909213325,1.6907051346,1.6905353139,1.6903198239
,1.6898778229,1.6893801719,1.6888803058,1.6883813394,1.6878855928,1.6874245827,1.6869798563,1.6865358888,1.6860927183,1.6856503434
]

In [ ]:
t = 1


i_up_uz = [0]
theta_h3 = [0]
t_alpha= [0] 
t_atm = [0] 
gwl_up = [0] 
gwl_low = [0]
theta_eq = [0] 
capris_max = [0]
p_gw = [0]


# Give initial theta_uz.
theta_uz_t0 = SoilSelector(7, 1, 1.7)['moist_cont_eq_ rz[mm]'].values # 1.5m is initial gwl.
theta_uz = [theta_uz_t0]

# Specify the parameter or use the default setting.
m = UnsaturatedZone(theta_uz_t0, soiltype = 7, croptype = 1)

while t <= iters-1:
    # only loop sol(), not repeat creating new object.
    sol = m.sol(i_uz[t], r_meas[t], e_ref[t], prev_gwl = gwl[t-1], delta_t = 1/24)
    
    
    i_up_uz.append(sol[0])
    theta_h3.append(sol[1])
    t_alpha.append(sol[2]) 
    t_atm.append(sol[3]) 
    gwl_up.append(sol[4])
    gwl_low.append(sol[5])
    theta_eq.append(sol[6])
    capris_max.append(sol[7])
    p_gw.append(sol[8])
    theta_uz.append(sol[9])

    # print('time step', t)
    t += 1
    
filename = 'Results_UnsaturatedZone_c2s2.csv'
np.savetxt('sol/' + filename, np.c_[i_up_uz, theta_h3, t_alpha, t_atm, gwl_up, gwl_low, theta_eq, capris_max, p_gw, theta_uz], fmt = "%.8f", delimiter=',', header = 'i_up_uz, theta_h3, t_alpha, t_atm, gwl_up, gwl_low, theta_eq, capris_max, p_gw, theta_uz') 

# Insert the Date column for locating purposes.
df = pd.read_csv('sol/' + filename)
df.insert(0, 'Date', date)
df.to_csv('sol/' + filename)

print('The results have been validated.')

### Conclusion.


1. Soil type and crop type as well as the groundwater level determine some parameters to be chosen. 
2. Once the soil type and crop type are chosen, the root zone thickness is then automatically defined. The combination of these three are in bas-input, you can not have abitrary rootzone thickness given a specific soil type and crop type.